# Noise reduction with PESQ (Google Colab or Kaggle)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/victor-merino-hub/audiolab/blob/main/notebooks/denoising_pesq.ipynb)

Runs `scripts/evaluate_denoising.py` on the VoiceBank+DEMAND test set **with PESQ**, which needs a C
compiler to install and therefore runs here instead of on Windows. No GPU needed: everything is
frame-by-frame DSP on the CPU.

- **Colab**: *Runtime → Run all* (the default CPU runtime is fine).
- **Kaggle**: *Settings → Internet → On*. The last cell does not apply there: the CSV appears under
  *Output* (`/kaggle/working`).

## 1. Code and metrics

In [ ]:
!git clone -q https://github.com/victor-merino-hub/audiolab.git
%cd audiolab
!pip install -q -e ".[pesq]"
import pesq  # fails here if the package did not compile

## 2. Test set (~310 MB, CC BY 4.0, Valentini-Botinhao et al., University of Edinburgh)

In [ ]:
# https://datashare.ed.ac.uk/handle/10283/2791 (direct file links of the repository's API)
!mkdir -p data/voicebank
!wget -q -O data/voicebank/clean.zip https://datashare.ed.ac.uk/server/api/core/bitstreams/dec213d3-bf57-4777-9663-c24bdce92d5e/content
!wget -q -O data/voicebank/noisy.zip https://datashare.ed.ac.uk/server/api/core/bitstreams/13c1bfbf-14a6-41db-9b41-8f7310f01ad5/content
!ls -la data/voicebank && file data/voicebank/*.zip
!cd data/voicebank && unzip -q clean.zip && unzip -q noisy.zip && rm clean.zip noisy.zip

# Stop here if the download did not work (both folders must hold the 824 sentences)
from pathlib import Path
counts = {d: len(list(Path("data/voicebank", d).glob("*.wav"))) for d in ("clean_testset_wav", "noisy_testset_wav")}
print(counts)
assert all(n == 824 for n in counts.values()), "test set incomplete: check the download above"

## 3. Evaluate: PESQ for every method and frame length

SI-SDR and STOI are computed locally; here only PESQ, to keep the run short (Colab has 2 CPUs).

In [ ]:
import os
print(os.cpu_count(), "CPUs")
!python scripts/evaluate_denoising.py --metrics pesq --out data/results/denoising_pesq.csv

## 4. Download the results

In [ ]:
from google.colab import files
files.download("data/results/denoising_pesq.csv")